# Continuous-Time Range-Only Localization with WNOA

In this example, we estimate a 3D trajectory from range measurements to known landmarks using a white-noise-on-acceleration (WNOA) Gaussian process prior. We use `AugmentedPoint3` for the position and a 3D vector for the velocity. Range measurements taken between estimated states are incorporated using `WnoaInterpFactor`.

See the [WNOA example on SE(3)](https://borglab.github.io/gtsam/gaussianprocesswnoainterpolationse3/) for an introduction to continuous-time estimation in GTSAM.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415
All Rights Reserved
Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import gtsam_unstable
import plotly.graph_objects as go
from gtsam import StateData
from gtsam.symbol_shorthand import V, X

## Setup and Ground Truth

We define a trajectory over four seconds, with estimated states at one-second intervals. Range measurements are available at the start of the trajectory and at three intermediate times in each interval. We use four known landmarks and add Gaussian noise to the ranges.

In [3]:
rng = np.random.default_rng(42)

knot_times = np.arange(0.0, 5.0)  # [0, 1, 2, 3, 4] s
measurement_times = np.array([t + f for t in knot_times[:-1] for f in (0.25, 0.5, 0.75)])

landmarks = np.array([[0.0, 0.0, 0.0],
                      [4.0, 0.0, 0.0],
                      [0.0, 4.0, 0.0],
                      [2.0, 2.0, 3.0]])

range_sigma = 0.02  # Standard deviation of each range measurement [m].


def true_position(t):
    return np.array([0.8 * t, 1.5 + np.sin(0.8 * t), 1.0 + 0.3 * np.cos(0.6 * t)])


def true_velocity(t):
    return np.array([0.8, 0.8 * np.cos(0.8 * t), -0.18 * np.sin(0.6 * t)])


print(f"{len(knot_times)} knots, {len(measurement_times)} measurement times, {len(landmarks)} landmarks")

5 knots, 12 measurement times, 4 landmarks


## Defining the States

We use `StateData` to associate a position key and a velocity key with each timestamp. Only the knot states are optimized; the measurement-time states are used to construct the interpolation factors.

In [4]:
knot_states = [StateData(X(k), V(k), t) for k, t in enumerate(knot_times)]
measurement_states = [StateData(X(100 + j), V(100 + j), t) for j, t in enumerate(measurement_times)]


def bracketing_knots(t):
    # Return the two knots that surround time t.
    k = int(np.searchsorted(knot_times, t, side="right")) - 1
    k = min(k, len(knot_states) - 2)
    return knot_states[k], knot_states[k + 1]

## Defining the Factor Graph

We first add a `WnoaMotionFactorAugmentedPoint3` between each pair of consecutive knots. The WNOA prior assumes zero-mean white noise on acceleration with power spectral density $Q_c$:

$$
\ddot{\mathbf{p}}(t) = \mathbf{w}(t), \qquad
\mathbb{E}[\mathbf{w}(t)\mathbf{w}(t')^\top] = Q_c\delta(t-t').
$$

For $\Delta t = t_{k+1}-t_k$, the motion residual and its covariance for axis $i$ are

$$
\mathbf{e}_k = \begin{bmatrix} \mathbf{p}_{k+1}-\mathbf{p}_k-\Delta t\,\mathbf{v}_k \\ \mathbf{v}_{k+1}-\mathbf{v}_k \end{bmatrix},
\qquad
Q_{k,i} = Q_{c,i}\begin{bmatrix} \Delta t^3/3 & \Delta t^2/2 \\ \Delta t^2/2 & \Delta t \end{bmatrix}.
$$

The motion factor takes two `StateData` objects and the diagonal of $Q_c$.

In [5]:
Qc = np.full(3, 0.1)  # Diagonal of the WNOA power spectral density.

graph = gtsam.NonlinearFactorGraph()
for left, right in zip(knot_states[:-1], knot_states[1:]):
    graph.add(gtsam_unstable.WnoaMotionFactorAugmentedPoint3(left, right, Qc))

print(f"{graph.size()} motion factors")

4 motion factors


### Squared-range measurements

The `KnownLandmarkSquaredRangeFactorAugmentedPoint3` factor uses the residual

$$
e = \|\mathbf{p}-\mathbf{l}\|^2-d^2,
$$

where $\mathbf{l}$ is the landmark position and $d$ is the measured range. We approximate the squared-range standard deviation by $2d\sigma$, where $\sigma$ is the range standard deviation.

We add the ranges at $t=0$ directly to the first knot to constrain its position.

In [6]:
def squared_range_factor(key, landmark, measured_range):
    noise = gtsam.noiseModel.Isotropic.Sigma(1, 2.0 * measured_range * range_sigma)
    return gtsam_unstable.KnownLandmarkSquaredRangeFactorAugmentedPoint3(
        key, landmark, measured_range, noise)


def measure_range(t, landmark):
    return np.linalg.norm(true_position(t) - landmark) + rng.normal(0.0, range_sigma)


for landmark in landmarks:
    graph.add(squared_range_factor(knot_states[0].pose, landmark, measure_range(0.0, landmark)))

### Ranges at intermediate times

For a measurement at $t_k<\tau<t_{k+1}$, we use `WnoaInterpFactorAugmentedPoint3` to wrap the squared-range factor. The interpolated state is given by the GP conditional mean

$$
\begin{bmatrix}\mathbf{p}_\tau\\\mathbf{v}_\tau\end{bmatrix}
=\Lambda(\tau)\begin{bmatrix}\mathbf{p}_k\\\mathbf{v}_k\end{bmatrix}
+\Psi(\tau)\begin{bmatrix}\mathbf{p}_{k+1}\\\mathbf{v}_{k+1}\end{bmatrix}.
$$

The resulting factor depends on the position and velocity keys of the neighboring knots. We set `fixed_noise_model=True` to retain the measurement noise model without accounting for interpolation uncertainty.

In [7]:
interp_measurements = []  # (state, landmark, measured range, factor), kept for a later check.
for state in measurement_states:
    left, right = bracketing_knots(state.time)
    for landmark in landmarks:
        measured_range = measure_range(state.time, landmark)
        inner = squared_range_factor(state.pose, landmark, measured_range)
        factor = gtsam_unstable.WnoaInterpFactorAugmentedPoint3(
            inner, {left, right}, {state}, Qc, True, True)
        interp_measurements.append((state, landmark, measured_range, factor))
        graph.add(factor)

key_name = lambda key: gtsam.Symbol(key).string()
example = interp_measurements[0][3]
print("Inner factor key:     ", key_name(measurement_states[0].pose))
print("Interp factor keys:   ", sorted(key_name(k) for k in example.keys()))
print("All graph keys:       ", sorted(key_name(k) for k in graph.keyVector()))
print(f"{graph.size()} factors in total, {len(interp_measurements)} of them interpolated ranges")

Inner factor key:      x100
Interp factor keys:    ['v0', 'v1', 'x0', 'x1']
All graph keys:        ['v0', 'v1', 'v2', 'v3', 'v4', 'x0', 'x1', 'x2', 'x3', 'x4']
56 factors in total, 48 of them interpolated ranges


## Optimization

The graph contains five position and five velocity variables. We perturb the ground-truth knot states to initialize the trajectory and solve using Levenberg–Marquardt.

In [8]:
values_init = gtsam.Values()
for state in knot_states:
    position = true_position(state.time) + rng.normal(0.0, 0.3, 3)
    gtsam_unstable.insertAugmentedPoint3(state.pose, gtsam_unstable.AugmentedPoint3(position), values_init)
    values_init.insert(state.velocity, true_velocity(state.time) + rng.normal(0.0, 0.2, 3))

params = gtsam.LevenbergMarquardtParams()
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, values_init, params)
result = optimizer.optimize()

print(f"Initial error: {graph.error(values_init):.4f}")
print(f"Final error:   {graph.error(result):.4f}  ({optimizer.iterations()} iterations)")
print(f"Optimized variables: {result.size()}")


def knot_positions(values):
    return np.array([gtsam_unstable.atAugmentedPoint3(s.pose, values).vector() for s in knot_states])


def knot_velocities(values):
    return np.array([values.atVector(s.velocity) for s in knot_states])


true_knots = np.array([true_position(t) for t in knot_times])
rmse = lambda estimate, truth: np.sqrt(np.mean(np.sum((estimate - truth) ** 2, axis=1)))
print(f"Knot position RMSE: initial {rmse(knot_positions(values_init), true_knots):.4f} m, "
      f"final {rmse(knot_positions(result), true_knots):.4f} m")

Initial error: 4456.4147
Final error:   11.7758  (4 iterations)
Optimized variables: 10
Knot position RMSE: initial 0.4758 m, final 0.0190 m


## Sampling the Continuous-Time Trajectory

We can now query the estimated trajectory between knots. For WNOA in a vector space, the state transition and covariance for one axis are

$$
\Phi(t,s)=\begin{bmatrix}1&t-s\\0&1\end{bmatrix},\qquad
Q(t,s)=q_c\begin{bmatrix}(t-s)^3/3&(t-s)^2/2\\(t-s)^2/2&t-s\end{bmatrix}.
$$

The interpolation matrices are

$$
\begin{aligned}
\Psi(\tau)&=Q(\tau,t_k)\Phi(t_{k+1},\tau)^\top Q(t_{k+1},t_k)^{-1},\\
\Lambda(\tau)&=\Phi(\tau,t_k)-\Psi(\tau)\Phi(t_{k+1},t_k).
\end{aligned}
$$

During optimization, `WnoaInterpFactorAugmentedPoint3` uses the C++ interpolator. Since the standalone interpolation utilities are not wrapped for `AugmentedPoint` in Python, we use NumPy to sample the trajectory for plotting. The common factor $q_c$ cancels from the interpolation matrices.

In [9]:
def wnoa_conditional_mean(values, t):
    # Position and velocity at time t from the two bracketing knots of `values`.
    left, right = bracketing_knots(t)
    dt, s = right.time - left.time, t - left.time
    Phi = lambda h: np.array([[1.0, h], [0.0, 1.0]])
    Q = lambda h: np.array([[h**3 / 3, h**2 / 2], [h**2 / 2, h]])  # Qc cancels.
    Psi = Q(s) @ Phi(dt - s).T @ np.linalg.inv(Q(dt))
    Lam = Phi(s) - Psi @ Phi(dt)

    xk = np.vstack([gtsam_unstable.atAugmentedPoint3(left.pose, values).vector(), values.atVector(left.velocity)])
    xk1 = np.vstack([gtsam_unstable.atAugmentedPoint3(right.pose, values).vector(), values.atVector(right.velocity)])
    x_tau = Lam @ xk + Psi @ xk1  # Rows: position, velocity; columns: axes.
    return x_tau[0], x_tau[1]


sample_times = np.linspace(knot_times[0], knot_times[-1], 161)
interp_init = np.array([wnoa_conditional_mean(values_init, t)[0] for t in sample_times])
interp_opt = np.array([wnoa_conditional_mean(result, t)[0] for t in sample_times])
truth = np.array([true_position(t) for t in sample_times])

We check the sampled positions against the GTSAM interpolation factors by comparing the unwhitened squared-range residuals at the measurement times.

In [10]:
max_diff = 0.0
for state, landmark, measured_range, factor in interp_measurements:
    gtsam_residual = factor.unwhitenedError(result)[0]
    p_tau, _ = wnoa_conditional_mean(result, state.time)
    numpy_residual = np.sum((p_tau - landmark) ** 2) - measured_range**2
    max_diff = max(max_diff, abs(gtsam_residual - numpy_residual))

print(f"Max |GTSAM interpolated residual - NumPy residual| over {len(interp_measurements)} factors: {max_diff:.2e}")

Max |GTSAM interpolated residual - NumPy residual| over 48 factors: 3.55e-15


## Results

The initial and optimized trajectories are interpolated from their respective knots. The markers show the optimized knots and the known landmarks.

In [11]:
fig = go.Figure()
fig.add_scatter3d(**dict(zip("xyz", truth.T)), mode="lines", name="Ground truth", line=dict(color="black", width=4))
fig.add_scatter3d(**dict(zip("xyz", interp_init.T)), mode="lines", name="Initial", line=dict(color="gray", dash="dash"))
fig.add_scatter3d(**dict(zip("xyz", interp_opt.T)), mode="lines", name="Optimized", line=dict(color="royalblue", width=5))
fig.add_scatter3d(**dict(zip("xyz", knot_positions(result).T)), mode="markers", name="Knots", marker=dict(color="royalblue", symbol="diamond", size=4))
fig.add_scatter3d(**dict(zip("xyz", landmarks.T)), mode="markers", name="Landmarks", marker=dict(color="crimson", symbol="x", size=4))
fig.update_layout(scene=dict(aspectmode="data", xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]"), height=550)
points = np.vstack([truth, interp_init, interp_opt, landmarks])
lo, hi = points.min(axis=0), points.max(axis=0)
pad = 0.08 * np.maximum(hi - lo, 1.0)
fig.update_layout(scene={f"{axis}axis": dict(range=[lo[i] - pad[i], hi[i] + pad[i]]) for i, axis in enumerate("xyz")})
fig.show()

We compare the position error along the initial and optimized trajectories. The vertical lines indicate the knot times.

In [12]:
err_init = np.linalg.norm(interp_init - truth, axis=1)
err_opt = np.linalg.norm(interp_opt - truth, axis=1)

fig = go.Figure()
fig.add_scatter(x=sample_times, y=err_init, name="Initial", line=dict(color="gray", dash="dash"))
fig.add_scatter(x=sample_times, y=err_opt, name="Optimized", line=dict(color="royalblue"))
for t in knot_times: fig.add_vline(x=t, line=dict(color="lightgray", dash="dot"))
fig.update_layout(xaxis_title="Time [s]", yaxis_title="Position error [m]", yaxis_type="log", height=350)
fig.show()

print(f"Trajectory RMSE: initial {rmse(interp_init, truth):.4f} m, optimized {rmse(interp_opt, truth):.4f} m")
print(f"Max position error after optimization: {err_opt.max():.4f} m")

Trajectory RMSE: initial 0.4312 m, optimized 0.0164 m
Max position error after optimization: 0.0276 m
